# Рекомендательные системы

Основано на материалах курса Д. И. Игнатова "Рекомендательные системы"

[Глава](https://education.yandex.ru/handbook/ml/article/intro-recsys) про рекомендательные системы от Яндекса



## Что такое рекомендательная система?




У нас есть множество пользователей. Для каждого пользователя есть множество объектов, которые он оценил (поставил рейтинг). Задачей рекомендательной системы является предсказание этого рейтинга для новых для пользователя объектов. Потом объекты с наиболее высокой оценкой будут предложены пользователю в ленте рекомендаций.

Например, мы можем предсказывать:
- оценку для фильма на сайте онлайн-кинотеатра;
- оценку для книги на сайте электронной библиотеки;
- просмотрит ли человек короткий ролик до конца;
- прослушает ли до конца песню;
- и т.д.

Более формально: для каждой пары пользователь-объект оценить рейтинг и предсказать топ наиболее "релевантных" пользователю объектов.

При этом оценка от пользователя обычно делится на два вида:
- __Явная__, когда пользователь точно выразил свое отношение к объекту (поставил лайк, написал отзыв)
- __Неявная__, когда мы вынуждены по косвенным признакам определять оценки (время просмотра видео, клики)

***

Бывает, что задачу ставят в более общем виде как задачу предсказания наличия ребер в двудольном графе. Тогда мы считаем, что у нас есть:
- Два набора объектов (обычно) различной природы (авторы и статьи, покупатели и продукты, книги и ключевые слова). При этом принято один набор называть объектами, а второй $-$ атрибутами.
- Информация о наличии связей между объектами. Причем связи могут быть только между объектами из разных наборов.


<img src="https://www.researchgate.net/publication/369199718/figure/fig4/AS:11431281183703059@1692966061534/K-partite-graph-illustration-a-A-user-item-bipartite-graph-with-nodes-representing-user.png" width=600 alt="rec">


И здесь есть два варианта постановки задачи:
- Object-Attribute task: предсказываем, должен ли данный объект иметь связь с данным атрибутом (является ли человек автором статьи, купит ли клиент продукт)
- Object-Object task: предсказываем, должны ли два объекта иметь связь с одним атрибутом (могут ли два человека быть соавторами, будут ли два продукта куплены вместе)


## Как выглядят данные для обучения?

Чаще всего данные представляют собой матрицу,
где по строкам и столбцам расположены два набора объектов, а в ячейках стоят оценки:

| user_id <br> item_id | 1 | 2 | 3 | 4 | 5 |
| -- | -- | -- | -- | -- | -- |
| 1 | 5.0 | 0.0 | 0.0 | 3.0 | 3.0 |
| 2 | 4.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 3 | 0.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 4 | 0.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 5 | 4.0 | 3.0 | 0.0 | 0.0 | 0.0 |


Иногда это длинная форма такой матрицы:

| user_id | item_id | rating |
| -- | -- | -- |
| 1 | 1 | 5.0 |
| 1 | 4 | 3.0 |
| 1 | 5 | 3.0 |
| 2 | 1 | 4.0 |
| 5 | 1 | 4.0 |
| 5 | 2 | 3.0 |

Хорошие датасеты:
- [GroupLens](https://grouplens.org/datasets/movielens/)
- [RecSys Challenge](https://www.recsyschallenge.com/2024/)
- [Kinopoisk's movies reviews](https://www.kaggle.com/datasets/mikhailklemin/kinopoisks-movies-reviews): русскоязычный и подходит для content-based решений
- [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) - англоязычный, подходит для content-based

## Методы оценки

Так как мы фактически решаем две задачи: предсказание наличия связи (и это классификация) и предсказание силы связи $-$ оценки (это регресиия), мы имеем два набора метрик:
1. MSE, MAE etc $-$ для предсказания оценки
2. Precision, Recall, F-score $-$ для предсказания связи

Также мы можем использовать метрики
- для сравнения множеств (коэффициент Жаккара)

$J(A, B) = \LARGE{\frac{\lvert A\cap B \rvert}{\lvert A \cup B \rvert}}$, например, чтобы сравнить списки реально просмотренных и рекомендованных товаров;

- корреляционные оценки (коэффициент Пирсона)

$r = \LARGE{\frac{\sum_{i=1}^n (x_i - \bar{x}) \times (y_i - \bar{y})}{\sqrt{{\sum_{i=1}^n (x_i - \bar{x}) ^ 2} \times {\sum_{i=1}^n (y_i - \bar{y}) ^ 2}}}}$, например, чтобы посчитать корреляцию между признаками контента и его востребованностью у пользователя;

- и метрики для ранжирования, например precision@k отражает, сколько релевантных элементов оказалось в топ-k лучших предсказаний $$Precision@k = \frac{|\text{релевантные документы в топ-}k|}{k}$$



## Подходы

#### Content-based

Здесь мы делаем предсказания на основе похожести объектов с точки зрения контента. При этом под контентом может пониматься как реальное соджержание объекта (текст книги / песни, картинка), так и метаинформация о нем (актеры и кассовые сборы, описание картинки или песни).

Обычно у данного подхода выделяют несколько недостатков:
- большая вычислительная сложность (тяжело работать с музыкой или большими текстами);
- для качественного анализа контента нужны сложные доменные модели (модели, которые обучены на данных конкретной предметной области и понимают её специфику). Кроме того, мы не можем считать картинку и текст похожими объектами, если у нас нет мультимодальных моделей;
- для многих типов объектов не очень понятно, что такое контент в целом;
- нужно много данных для первичного обучения.

#### Collaborative filtering

Здесь есть два набора подходов:
1. Item-based: рекомендуем новые объекты на основе их похожести на выбранные пользователем ранее. Какие здесь могут быть сложности? Что значит "похожесть" в данном случае?

2. User-based: рекомендуем пользователю новые объекты на основе того, что выбирают похожие на него пользователи. Какие здесь могут быть сложности? Что значит "похожесть" в данном случае?

<img src="https://www.researchgate.net/profile/Prakash-Upadhyaya/publication/366902172/figure/fig2/AS:11431281111439652@1672973053129/User-based-and-Item-based-Collaborative-Filtering.png" alt="rec"/>

На практике часто используются комбинированные методы.

## Возможные алгоритмы решения

#### 1. Методы на основе векторного расстояния между пользователями / объектами

У нас есть квадратная матрица с рейтингами:

| user_id <br> item_id | 1 | 2 | 3 | 4 | 5 |
| -- | -- | -- | -- | -- | -- |
| 1 | 5.0 | 0.0 | 0.0 | 3.0 | 3.0 |
| 2 | 4.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 3 | 0.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 4 | 0.0 | 0.0 | 0.0 | 0.0 | 0.0 |
| 5 | 4.0 | 3.0 | 0.0 | 0.0 | 0.0 |

Давайте считать, что строки $-$ это вектора пользователей (тогда каждый признак $-$ это оценка для фильма), а столбцы $-$ это вектора объектов (признак $-$ оценка от пользователя). Тогда мы можем находить ближайших пользователей / объекты на основе любого метода подсчета векторной близости (MSE, cosine distance, etc.)

#### 2. Методы на основе FCA (Formal Concept Analysis) или pattern mining

Существует группа методов, позволяющих находить множества похожих объектов на основе графового представления данных (например, берем все группы объектов, имеющих связи с одними и теми же вершинами).

Тогда мы считаем похожими объекты или пользователей, оказавшихся в одном множестве.

-  Матричные разложения (и в целом весь спектр методов для задачи восстановления пропусков в матрице)

- NLP подходы (считаем объекты/пользователей токенами и учимся их векторизовать)

## Практика

В питоне есть библиотека `surprise` ([документация](https://surprise.readthedocs.io/en/stable/index.html)), в которой собраны многие методы и датасеты для рексистем

In [1]:
!pip install numpy==1.26.4 -q
!pip install scikit-surprise -q
# после этого надо перезапустить сессию в Colab

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 37.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible

In [1]:
from surprise import NormalPredictor, KNNBasic, KNNWithMeans, SVD
from surprise import Dataset
from surprise.model_selection import cross_validate

import pandas as pd

Прочитаем датасет. Заметим, что это какой-то сложный класс, который не умеет сам себя красиво рисовать

In [2]:
data = Dataset.load_builtin('ml-100k')
data

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k


Но мы можем посмотреть на его содержание в сыром виде. Здесь есть четыре колонки:
- ID пользователя
- ID объекта (фильма)
- Ретинг, который пользователь поставил фильму
- Время, когда оценка была поставлена

__Зачем нам время?__

В задачах рекомендаций, как и в любой задаче, где есть деление на "сейчас" и "будущее", имеет смысл разделение на обучение и тест делать не случайно, а по времени.

In [4]:
df = pd.DataFrame(data.raw_ratings,
                  columns=['user_id', 'item_id', 'rating', 'timestamp'])
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


Теперь попробуем решить задачу предсказания рейтинга.

Начнем с `NormalPredictor`. Этот алгоритм просто предсказывает случайный рейтинг на основе нормального распеделения. Это будет наш baseline.

In [5]:
# Создаем класс, у него никаких дополнительных параметров нет
algo = NormalPredictor()

# Запускаем кросс-валидацию
cv_res = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm NormalPredictor on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.5285  1.5141  1.5233  1.5284  1.5214  1.5231  0.0053  
MAE (testset)     1.2270  1.2160  1.2236  1.2248  1.2206  1.2224  0.0038  
Fit time          0.22    0.19    0.24    0.12    0.12    0.18    0.05    
Test time         0.17    0.41    0.16    0.17    0.08    0.20    0.11    


Функция возвращает нам словарь с результатами:

In [6]:
cv_res

{'test_rmse': array([1.52845584, 1.51406133, 1.52329755, 1.52838855, 1.52142758]),
 'test_mae': array([1.22703456, 1.21598808, 1.22363997, 1.22481968, 1.22062045]),
 'fit_time': (0.22011566162109375,
  0.19025564193725586,
  0.24068045616149902,
  0.11839914321899414,
  0.11638188362121582),
 'test_time': (0.17342519760131836,
  0.409360408782959,
  0.15833139419555664,
  0.16698861122131348,
  0.07666611671447754)}

Теперь попробуем `KNNBasic` $-$ это алгоритм, который использует векторную близость между объектами / пользователями. У него есть следующие параметры:
- `k` $-$ максимальное кол-во ближайших объектов, для которых мы усредняем оценки (если всего оценок больше, то все остальные просто игнорируем);
- `min_k` $-$ минимальное кол-во ближайших объектов, для которых мы усредняем оценки (если меньше, то берем просто среднее по обучающим данным);
- `sim_options` $-$ всякие параметры для функции близости, здесь из важных:
    - `user_based` $-$ `True` или `False`, определяет какой вариант алгоритма мы используем;
    - `name` $-$ название метрики близости.

Попробуем сначала `user-based` подход.

Допустим, мы хотим предсказать оценку кинолюбителя Аркадия (user=1) для фильма "Матрица" (item=50).

У нас есть оценки других пользователей:
- пользователь_2: ставил похожие оценки на общие фильмы с Аркадием (похож на 80%)
- пользователь_4: похож на 10%
- пользователь_7: похож на 60%

Берем k самых похожих пользователей (соседей). Кто из наших соседей смотрел "Матрицу"?
- пользователь_2: поставил 5
- пользователь_4: не смотрел (пропускаем)
- пользователь_7: поставил 4

Вычисляем средневзвешенное значение:
rating = (5 * 0.8 + 4 * 0.6) / (0.8 + 0.6)

In [7]:
algo = KNNBasic(k=40, min_k=1, sim_options={'user_based': True})
cv_res = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9822  0.9750  0.9803  0.9797  0.9760  0.9786  0.0027  
MAE (testset)     0.7736  0.7692  0.7745  0.7736  0.7737  0.7729  0.0019  
Fit time          0.47    0.48    0.50    0.53    0.49    0.49    0.02    
Test time         3.08    3.22    4.45    3.30    3.27    3.46    0.50    


А теперь item-based:

In [8]:
algo = KNNBasic(k=40, min_k=1, sim_options={'user_based': False})
cv_res = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9745  0.9754  0.9745  0.9713  0.9720  0.9735  0.0016  
MAE (testset)     0.7684  0.7703  0.7721  0.7702  0.7660  0.7694  0.0021  
Fit time          0.89    0.77    0.71    0.70    0.78    0.77    0.07    
Test time         4.25    3.69    4.63    3.69    3.77    4.01    0.37    


Алгоритм выше во многом хорош, но на практике чаще используется его улучшенная версия. В библиотеке она называется `KNNWithMeans`.

В чем идея улучшений?

Очевидно, что все пользователи имеют у себя в голове немного разную шкалу (кто-то ставит только 5-ки, а кто-то никогда не ставит ничего выше 4-х), так же как и у объекта обычно рейтинги тяготеют к какому-то определенному значению. Чтобы учитывать подобные нюансы, в формулу для усреднения вводится нормирование на среднее значение по пользователю / объекту.

Вернёмся к кинолюбителю Аркадию.

Средняя оценка Аркадия = 3.8

А вот его соседи и их коэфф. сходства, оценка "Матрицы" и средняя оценка по всем фильмам.
- Петя: sim = 0.9, rating = 5, mean = 4.5
- Маша: sim = 0.7, rating = 4, mean = 3.0

Центрируем оценки соседей:
- Петя: 5 - 4.5 = +0.5
- Маша: 4 - 3.0 = +1

Считаем взвешенное среднее центрированных оценок:
mean_centered = (0.5 * 0.9 + 1 * 0.7) / (0.9 + 0.7) ≈ 0.72

Добавляем среднее Аркадия:
rating = 3.8 + 0.72 ≈ 4.5

In [9]:
algo = KNNWithMeans(k=40, min_k=1, sim_options={'user_based': True})
cv_res = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNWithMeans on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9499  0.9452  0.9507  0.9571  0.9504  0.9507  0.0038  
MAE (testset)     0.7498  0.7450  0.7495  0.7533  0.7477  0.7490  0.0027  
Fit time          0.79    0.58    0.68    0.48    0.52    0.61    0.11    
Test time         6.36    3.53    4.83    3.41    3.30    4.28    1.18    


Еще можно проверить, как сработает алгоритм, основанный на SVD разложении.

Общая идея:

- Делаем разложение через градиентный спуск на две (а не три, как при классическом SVD) матрицы, считаем, что одна из них содержит вектора пользователей, а другая $-$ объектов (внутренняя размерность — это факторы — скрытые характеристики, которые алгоритм сам находит в данных).
- Тогда рейтинг $-$ это скалярное произведение вектора пользователя на вектор объекта (собственно, мы матрицу для этого и раскладывали)

Подробнее можно прочитать [тут](https://habr.com/ru/companies/surfingbird/articles/139863/).

In [ ]:
algo = SVD()
cv_res = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9280  0.9477  0.9309  0.9322  0.9372  0.9352  0.0069  
MAE (testset)     0.7323  0.7444  0.7359  0.7367  0.7378  0.7374  0.0040  
Fit time          1.44    1.47    1.86    1.67    1.44    1.58    0.17    
Test time         0.23    0.15    0.20    0.11    0.12    0.16    0.05    


По-хорошему, чтобы качественно оценить качество предсказаний, нужно сначала разделить данные на train и test и запускать кросс-валидацию только на train, а потом оценить качество на test.

Так как рекомендательные системы призваны делать прогнозы, будет логично поместить более поздние оценки в test, чтобы основывать прогнозы на более ранних оценках.

In [9]:
dir(data)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'build_full_trainset',
 'construct_testset',
 'construct_trainset',
 'has_been_split',
 'load_builtin',
 'load_from_df',
 'load_from_file',
 'load_from_folds',
 'ratings_file',
 'raw_ratings',
 'read_ratings',
 'reader']

In [3]:
raw = data.raw_ratings
print(len(raw))

# Сортируем по timestamp (последние оценки - самые свежие)
sorted_ratings = sorted(raw, key=lambda x: x[3])
sorted_ratings[:5]

100000


[('259', '255', 4.0, '874724710'),
 ('259', '286', 4.0, '874724727'),
 ('259', '298', 4.0, '874724754'),
 ('259', '185', 4.0, '874724781'),
 ('259', '173', 4.0, '874724843')]

In [4]:
# Определим размер сплита
test_size = int(len(sorted_ratings) * 0.2)
train_raw = sorted_ratings[:-test_size]
test_raw = sorted_ratings[-test_size:]
test_raw[-5:]

[('729', '333', 4.0, '893286638'),
 ('729', '748', 4.0, '893286638'),
 ('729', '313', 3.0, '893286638'),
 ('729', '300', 4.0, '893286638'),
 ('729', '272', 4.0, '893286638')]

In [5]:
from surprise import Reader
train_df = pd.DataFrame(train_raw, columns=['user', 'item', 'rating', 'timestamp'])
train_df = train_df[['user', 'item', 'rating']]
# Создаем Dataset
reader = Reader()
train_data = Dataset.load_from_df(train_df, reader=reader)
print(train_data.raw_ratings[:5])
print(type(train_data))

[('259', '255', 4.0, None), ('259', '286', 4.0, None), ('259', '298', 4.0, None), ('259', '185', 4.0, None), ('259', '173', 4.0, None)]
<class 'surprise.dataset.DatasetAutoFolds'>


In [6]:
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [20, 50],      # количество факторов
    'n_epochs': [10, 20],         # количество эпох
    'lr_all': [0.002, 0.005, 0.01],   # скорость обучения
    'reg_all': [0.02, 0.1]      # регуляризация
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=['rmse'],
    cv=3,                       #cross-validation
    n_jobs = -1,
    joblib_verbose=0
)

gs.fit(train_data)

In [7]:
gs.best_params

{'rmse': {'n_factors': 50, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.1}}

In [8]:
from surprise import accuracy

best_algo = gs.best_estimator['rmse']
trainset = train_data.build_full_trainset()
best_algo.fit(trainset)

testset = [(uid, iid, float(r)) for uid, iid, r, _ in test_raw]
predictions = best_algo.test(testset)
test_rmse = accuracy.rmse(predictions, verbose=False)
test_mae = accuracy.mae(predictions, verbose=False)

print(test_rmse, test_mae)

1.0214236379499748 0.8194172233676927
